# Programming exercise 10: Ground State Search with the Variational Monte Carlo Method

In [1]:
!pip install flax
!nvidia-smi
!pip install git+https://github.com/NiklasEuler/CQD_SS26.git

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
/bin/bash: nvidia-smi: command not found
Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
  Cloning https://github.com/NiklasEuler/CQD_SS26.git to /tmp/pip-req-build-txojse2e
  Running command git clone --filter=blob:none --quiet https://github.com/NiklasEuler/CQD_SS26.git /tmp/pip-req-build-txojse2e
  Resolved https://github.com/NiklasEuler/CQD_SS26.git to commit 9ee0b4fa8c7d2b473067d1c12d21842914ad98ef
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os
os.environ["JAX_PLATFORM_NAME"] = "cpu"

import sys
sys.path.append("../src")

import jax
import jax.numpy as jnp
import flax.linen as nn
import numpy as np
import time
import os
import scipy.sparse as sparse 
import scipy.sparse.linalg as sLA
import optax
import matplotlib.pyplot as plt
import Comp_Quant_Dynam as cqd

In [3]:
# Common parameters
J = 1.0                  # Ising interaction strength
B = 1.0                  # Transverse field strength
g = 0.5                  # Longitudinal field strength

N_spins = 10             # Number of spins in the chain
N_MC = 500               # Number of Monte Carlo samples
num_iterations = 200     # Number of optimization iterations
seed = 1                 # Random seed

lr = 0.005               # General learning rate for NQS / FFNN
lr_jastrow = 0.02        # Learning rate for Jastrow ansatz

### Exercise 1: The Jastrow Ansatz and Markov Chain Monte Carlo (MCMC)

We will start by constructing a short-range Jastrow ansatz that entangles nearest and next-to nearest neigbours,

$$\braket{\mathbf{s}|\psi_\text{jas}} = \exp\left(\sum\limits_{i}J_1 \sigma^z_i\sigma^z_{i+1} + J_2 \sigma^z_i\sigma^z_{i+2}\right),$$

where the parameters $J_1$ and $J_2$ have to be learned. We will construct the Jastrow ansatz using a `flax` based class. It is common to define the logarithm of the variational wave functions so we define `Jastrow()`$=\log \psi_\text{jas}$

- After defining the Jastrow ansatz, initialize the model and test the output on a spin state $\ket{111...111}$ with $N=10$. 

In order to do ground state search via variational Monte Carlo, we need a sampling algorithm that can generate random vectors of $N$ spins. But not all spin configurations are equally likely! That's why we use Markov Chain Monte Carlo (MCMC) to sample from the Born distribution of the variational wave function $p_{\mathbf{\theta}}(\mathbf{s}) = \frac{|{\psi_{\mathbf{\theta}}(\mathbf{s})}|^2}{\braket{\psi_{\mathbf{\theta}}|\psi_{\mathbf{\theta}}}}$. 

For this, we have implemented a sampler via the Metropolis-Hastings algorithm using `jax.lax.scan`. Make sure you understand, how the sampler works. Compare the code to the algorithm sketch below: At each step
- Start with configuration s
- Propose a new spin state s' by doing a random spin flip (0 -> 1 or 1 -> 0)
- Accept the proposed new spin state with acceptance probability

$$p_\text{accept}(s,s') = \min \left( 1 , \frac{p_{\mathbf{\theta}}(\mathbf{s'})}{p_{\mathbf{\theta}}(\mathbf{s})}\right)$$
- Otherwise keep s
- Repeat until $N_\text{samp}$ samples have been accepted.

Why is there a nested "full sweep" loop within the outer loop? Why is this important? Think about the independency and autocorrelations of samples. 

In [4]:
model = cqd.utility.Jastrow() # Create the Jastrow model
s = jnp.ones(N_spins ) # Define the spin state |111...111>

# Initialize parameters
key = jax.random.PRNGKey(0) 
params = model.init(key, s)

# Apply model to test state
log_psi = model.apply(params, s)

print("Initialized Jastrow parameters=\n", params)
print(" ")
print("log(psi) for |111...111> with N=10=\n", log_psi)

Initialized Jastrow parameters=
 {'params': {'J1': Array(0., dtype=float32), 'J2': Array(0., dtype=float32)}}
 
log(psi) for |111...111> with N=10=
 0.0


In [5]:
num_samples = 15

spin_samples = cqd.utility.MCMC_Sampler_Metropolis_Hastings(
    model=model,
    params=params,
    init_state=jnp.ones((N_spins,)),
    num_samples=num_samples,
    PRNGkey=jax.random.PRNGKey(7),
)

print(f"Let us look at {num_samples} spin state samples:")
print()
print(spin_samples)

Let us look at 15 spin state samples:

[[0. 1. 1. 0. 0. 0. 1. 0. 1. 0.]
 [1. 1. 0. 1. 1. 1. 1. 0. 0. 0.]
 [1. 1. 1. 0. 1. 0. 1. 1. 0. 0.]
 [1. 1. 1. 0. 1. 0. 0. 1. 1. 0.]
 [1. 0. 1. 0. 0. 1. 1. 1. 1. 0.]
 [0. 0. 1. 1. 0. 1. 0. 0. 1. 0.]
 [0. 1. 0. 1. 1. 0. 0. 0. 0. 1.]
 [1. 1. 1. 0. 1. 0. 0. 0. 1. 1.]
 [0. 1. 1. 0. 1. 1. 1. 1. 1. 1.]
 [1. 0. 0. 1. 0. 1. 0. 0. 1. 0.]
 [0. 0. 1. 1. 0. 0. 0. 1. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 1.]
 [1. 1. 0. 0. 1. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 1. 1. 0. 0. 1.]
 [1. 1. 0. 1. 1. 0. 1. 0. 1. 0.]]


In [6]:
# manual test with nonzero Jastrow parameters
params2 = {"params": {"J1": jnp.array(0.1), "J2": jnp.array(0.2),}}
log_psi2 = model.apply(params2, s)
print("log(psi) with J1=0.1 and J2=0.2=\n", log_psi2)

log(psi) with J1=0.1 and J2=0.2=
 3.0


The nested full-sweep loop is used because one Metropolis-Hastings update flips only one spin, so consecutive configurations would be very similar.

The outer loop saves the Monte Carlo samples, while the inner full-sweep loop performs about (N) spin-flip attempts before saving each sample.

This reduces autocorrelation between saved samples and makes them more approximately independent, which improves the reliability of Monte Carlo estimates.

### Exercise 2: Ground State Search via Stochastic Gradient Descent (SGD) 

Before we define our ground state search algorithm, let us define our minimization objective, the variational energy expectation value $E_\text{var}(\mathbf{\theta})$ which is given by,

$$
E_\text{var}(\mathbf{\theta}) =  \sum\limits_{s} \frac{|\psi_{\mathbf{\theta}}(s)|^2}{\braket{\psi_{\mathbf{\theta}}|\psi_{\mathbf{\theta}}}} E_\text{loc} (s) = \langle E_\text{loc} (s)\rangle_{s\sim p_{\mathbf{\theta}}},
$$
here $E_\text{loc} (s)$ is the local estimate of the energy that can be further written down as

$$
E_\text{loc} (s) = \frac{\bra{s}\hat{H}\ket{\psi_{\mathbf{\theta}}}}{\braket{s|\psi_{\mathbf{\theta}}}} = \sum\limits_{s'} \bra{s}\hat{H}\ket{s'} \frac{\psi_{\mathbf{\theta}}(s')}{\psi_{\mathbf{\theta}}(s)}.
$$

Now, we can implement the gradient of the variational energy $E_\text{var}(\mathbf{\theta})$ as the Monte Carlo averaged expression,

$$\nabla_{\theta_k}E_\text{var}(\mathbf{\theta}) = 2 \, \text{Re} \left[ \langle O^*_k(s) E_\text{loc} (s)\rangle_{s\sim p_{\mathbf{\theta}}} - \langle O^*_k(s)\rangle_{s\sim p_{\mathbf{\theta}}} \langle E_\text{loc} (s)\rangle_{s\sim p_{\mathbf{\theta}}} \right],$$

with the logarithmic variational derivatives $O_k (s) = \partial_{\theta_k} \log \psi_{\mathbf{\theta}}(s)$ and update the variational parameters accordingly via gradient descent,

$$\theta^{(i+1)} = \theta^{(i)} - \eta \nabla_{\theta_k}E_\text{var}(\mathbf{\theta}), $$

using the optax optimizer `optax.adam(learning_rate=lr)` or  `optax.adabelief(learning_rate=lr)` and the update functions `optimizer.update()` as well as `optax.apply_updates()`.

- Plot the variational ground state energy $E_\text{gs}(\mathbf{\theta})$ against the number of iterations and show its convergence by comparing to the exact ground state energy
- (Optional): Play around with different learning rates $\eta_i$ and number of MC samples $N_\text{MC}$ to see the effect on the speed of convergence and statistical fluctuations

In [ ]:
model = cqd.utility.Jastrow()

# Initial spin configuration
init_state = jnp.ones((N_spins,))

# Initialize model parameters
key = jax.random.PRNGKey(0)
params = model.init(key, init_state)

# Optimizer
optimizer = optax.adam(learning_rate=lr)
opt_state = optimizer.init(params)

# Exact ground-state energy 
E_exact = cqd.hamiltonians.E_TFIM_individual_exact(N_spins, B)
print("Exact ground state energy=\n", E_exact)

# Store energy values
energy_history = []

# Random key for Monte Carlo sampling
sample_key = jax.random.PRNGKey(123)

for it in range(num_iterations):

    # New random key for this iteration
    sample_key, subkey = jax.random.split(sample_key)

    # Generate Monte Carlo samples
    samples = cqd.utility.MCMC_Sampler_Metropolis_Hastings(
        model=model,
        params=params,
        init_state=init_state,
        num_samples=num_samples,
        PRNGkey=subkey,
    )

    # Continue the Markov chain from the last sample
    init_state = samples[-1]

    # Compute variational energy and gradient
    E_var, grads = cqd.utility.energy_and_gradient(
        params=params,
        samples=samples,
        model=model,
        B=B,
    )

    energy_history.append(float(E_var))

    # Optimizer update
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)

    if it % 20 == 0:
        print(f"Iteration {it:4d} | E_var = {float(E_var):.8f}")

Exact ground state energy=
 -12.784906442999324
Iteration    0 | E_var = -10.40000057
Iteration   20 | E_var = -12.59937096
Iteration   40 | E_var = -12.46662998
Iteration   60 | E_var = -12.51844978


In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(energy_history, label="VMC Jastrow energy")
plt.axhline(E_exact, linestyle="--", label="Exact ground_state energy")

plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title("Ground_State Search with Jastrow Ansatz")
plt.legend()
plt.grid(True)
plt.show()

The variational energy is estimated from Monte Carlo samples drawn from the Born distribution, and each sample is used to compute the local energy.

The average of the local energies, and the VMC covariance formula is used to compute the gradient with respect to the variational parameters.

The parameters are updated using `optax.adam`; the energy should decrease toward the exact ground-state energy, with fluctuations depending on the number of samples and learning rate.

### Exercise 3: Ground State Search using Neural Quantum States

Now we can build the Neural Quantum States (NQS) ansatz (https://www.science.org/doi/10.1126/science.aag2302) . Construct a feed-forward neural network (FFNN), with randomly initialized network parameters. For spin systems usually, the FFNN takes a vector of spins $\mathbf{\sigma}$ as inputs and return the logarithm of the spin wave function as an output, 

$$\log\psi_{\mathbf{\theta}}(\mathbf{\sigma}) = \frac{1}{2}\log \rho_{\mathbf{\theta}}(\mathbf{\sigma}) + i \phi_{\mathbf{\theta}}(\mathbf{\sigma}).$$

But here the ground state wave function of the 1D TFIM is purely real, so we can construct a FFNN model that outputs a single real number.

Choose a suited nonlinear activation function, and initialize the FFNN with random variational parameters $\mathbf{\theta}$ chosen from a normal distribution. The $l$-th layer of the Feed-Forward Neural Network consists of the weight matrix $W^{(l)}_{kl}$ and bias vector $b^{(l)}_l$ and the output is sent through a nonlinear activation function,

$$\text{FFNN}^{(l+1)}(\mathbf{s}) = \text{ActFunc}\left( \sum\limits_{k}W^{(l)}_{kl}s^{(l)}_k + b^{(l)}_l\right).$$

Initialize the weights $W^{(l)}_{kl}$ from a normal distribution and the biases $b^{(l)}_l$ from zero. `jax.nn.initializers.lecun_normal` and `jax.nn.initializers.zeros` functions may be helpful.

- Run the ground search algorithm now for the NQS ansatz and compare the convergence in a plot with the Jastrow ansatz using again the analytical ground state energy benchmark.
- How does the NQS ansatz perform for large and small network sizes and larger and smaller learning rates? What are potential reasons for this performance and possible ways to improve?

(Optional): In order to get an intuition of how the structure of the FFNN changes its performance you can play around with different hidden layer numbers and activation functions on https://playground.tensorflow.org, especially for regression problems.

In [ ]:
E_exact = cqd.hamiltonians.E_TFIM_individual_exact(N_spins, B)

jastrow_model = cqd.utility.Jastrow()

jastrow_params, jastrow_energies = cqd.utility.run_vmc_training(
    model=jastrow_model,
    N_spins=N_spins,
    B=B,
    N_MC=N_MC,
    num_iterations=num_iterations,
    lr=lr_jastrow,
    seed=seed,
)

In [ ]:
nqs_model = cqd.utility.NQS_FFNN(
    hidden_dims=(16,),
    actfunc=nn.tanh,
)

nqs_params, nqs_energies = cqd.utility.run_vmc_training(
    model=nqs_model,
    N_spins=N_spins,
    B=B,
    N_MC=N_MC,
    num_iterations=num_iterations,
    lr=lr,
    seed=seed,
)

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(jastrow_energies, label="Jastrow ansatz")
plt.plot(nqs_energies, label="NQS / FFNN ansatz")
plt.axhline(E_exact, linestyle="--", label="Exact ground-state energy")

plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title("Ground-State Search: Jastrow vs NQS")
plt.legend()
plt.grid(True)
plt.show()

The NQS ansatz uses a feed-forward neural network to represent \(\log \psi_\theta(s)\), so it is more flexible than the Jastrow ansatz.

Small networks may not represent enough correlations, while large networks can perform better but are harder to optimize and more sensitive to the learning rate.

Larger learning rates can make training unstable, smaller learning rates can converge slowly, and more Monte Carlo samples usually reduce statistical fluctuations.

### Exercise 4: Ground state of the tilted TFIM 

In this exercise we look at the tilted 1D TFIM,

$$
H=\sum_{i=0}^{N-1} -J\sigma_z^{(i)}\sigma_z^{(i+1)} - B \sigma_x^{(i)} - g \sigma_z^{(i)}.
$$

where we add a longitudinal field $g$ to the system which breaks the spin-flip symmetry. Thus, for this system there exists no analytical solution of the ground state energy and exact diagonalization is only possible to a certain number of spins. Here comes the advantage of variational methods truly into play. Define the Hamiltonian from above analogously to sheet 8 and solve the ground state energy using exact diagonalization roughly for spin numbers $N\in \{2,\dots, 20\}$ and find the same ground state via variational Monte Carlo with your model of choice. Then push the limits of what is possible with ED by using the variational model for finding the ground state energies for larger systems $N\in \{21,\dots, 100\}$, as long as your computer can handle it.


In [ ]:
N_values_ED = range(2, 21)
E_exact_list = []
N_exact_list = []

for N_spins in N_values_ED:
    try:
        E_exact = cqd.hamiltonians.E_tilted_TFIM_exact_ED(
            N=N_spins,
            J=J,
            B=B,
            g=g,)

        E_exact_list.append(E_exact)
        N_exact_list.append(N_spins)

        print(f"N = {N_spins:2d} | E_exact = {E_exact:.8f}")

    except Exception as error:
        print(f"Stopped at N = {N_spins}. Error:")
        print(error)
        break

In [ ]:
E_vmc_list = []
N_vmc_list = []

for N_spins in N_exact_list:
    print(f"\nRunning VMC for N = {N_spins}")

    model = cqd.utility.NQS_FFNN(
        hidden_dims=(16,),
        actfunc=nn.tanh,
    )

    params, energies = cqd.utility.run_vmc_training_tilted(
        model=model,
        N_spins=N_spins,
        J=J,
        B=B,
        g=g,
        N_MC=N_MC,
        num_iterations=num_iterations,
        lr=lr,
        seed=N_spins,
    )

    E_vmc_list.append(energies[-1])
    N_vmc_list.append(N_spins)

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(N_exact_list, E_exact_list, "o-", label="Exact diagonalization")
plt.plot(N_vmc_list, E_vmc_list, "s-", label="VMC / NQS")

plt.xlabel("Number of spins N")
plt.ylabel("Ground-state energy")
plt.title("Tilted TFIM: Exact Diagonalization vs VMC")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
large_N_values = [21, 30, 40, 60, 80, 100]

large_E_vmc = []
for N_spins in large_N_values:
    print(f"\nRunning large-system VMC for N = {N_spins}")

    model = cqd.utility.NQS_FFNN(
        hidden_dims=(32,),
        actfunc=nn.tanh,
    )

    params, energies = cqd.utility.run_vmc_training_tilted(
        model=model,
        N_spins=N_spins,
        J=J,
        B=B,
        g=g,
        N_MC=N_MC,
        num_iterations=num_iterations,
        lr=lr,
        seed=N_spins,
    )

    large_E_vmc.append(energies[-1])

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(large_N_values, large_E_vmc, "o-", label="VMC / NQS")

plt.xlabel("Number of spins N")
plt.ylabel("Variational ground-state energy")
plt.title("Tilted TFIM: VMC for Larger Systems")
plt.legend()
plt.grid(True)
plt.show()

The tilted TFIM contains an additional longitudinal field term \(-g\sum_i \sigma_i^z\), which breaks the spin-flip symmetry. Therefore, the model no longer has the same analytical ground-state solution as the standard 1D TFIM.

For small system sizes, exact diagonalization can be used to compute the ground-state energy. However, exact diagonalization becomes expensive because the Hilbert space dimension grows as \(2^N\).

For larger systems, VMC with a neural quantum state can still be used because it avoids storing the full wavefunction and instead optimizes a variational ansatz using Monte Carlo samples.

### Exercise 5: Testing the GPU Acceleration

Now, we want to test the true GPU acceleration that working with `jax` provides us. Therefore, go to Google Collab (https://colab.research.google.com), where you can upload this python notebook. Go to the upper right corner ("Connect") and choose to connect with device: T4 GPU. You need to first install the CQD package locally on to the cluster via:

`!pip install git+https://github.com/NiklasEuler/CQD_SS26.git `

Now, make sure that the line `os.environ["JAX_PLATFORM_NAME"] = "cpu"` is commented out and `jax` should automatically communicate with the GPU device, which is referred to as `CUDA device`.

- Make sure, that both the MCMC sampler and the ground state search functions are defined purely with `jax.lax` loops such as `jax.lax.scan` or `jax.lax.fori_loop` and that trivially parallelizable batch evaluations are vectorized via `jax.vmap`. And enforce `JIT` compilation by using `@jax.jit` infront of your loss function.
- Run the ground state search from above on the GPUs and compare the runtime difference. 
- Now that the code runs faster, you might want to do a ground state search for even higher number of spins. 

In [ ]:
!nvidia-smi
!pip install git+https://github.com/NiklasEuler/CQD_SS26.git

In [ ]:
# Test how many GPU devices are visible
print("Available devices:", jax.devices())

In [ ]:
def make_train_step(model, optimizer, N_MC, B):
    @jax.jit
    def train_step(params, opt_state, init_state, key):
        key, sample_key = jax.random.split(key)

        samples = cqd.utility.MCMC_Sampler_Metropolis_Hastings(
            model=model,
            params=params,
            init_state=init_state,
            num_samples=N_MC,
            PRNGkey=sample_key
        )

        new_init_state = samples[-1]

        E_var, grads = energy_and_gradient(
            params=params,
            samples=samples,
            model=model,
            B=B
        )

        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)

        return params, opt_state, new_init_state, key, E_var

    return train_step

In [ ]:
N_spins = 16
B = 1.0

N_MC = 200
num_iterations = 100
lr = 0.01

model = NQS_FFNN(hidden_dims=(32,), actfunc=nn.tanh)

key = jax.random.PRNGKey(0)
params = model.init(key, jnp.ones((N_spins,)))

optimizer = optax.adam(learning_rate=lr)
opt_state = optimizer.init(params)

init_state = jnp.ones((N_spins,))

train_step = make_train_step(
    model=model,
    optimizer=optimizer,
    N_MC=N_MC,
    B=B
)

In [ ]:
params, opt_state, init_state, key, E_var = train_step(
    params,
    opt_state,
    init_state,
    key
)

jax.block_until_ready(E_var)

In [ ]:
energies_gpu = []

start = time.perf_counter()

for it in range(num_iterations):
    params, opt_state, init_state, key, E_var = train_step(
        params,
        opt_state,
        init_state,
        key
    )

    jax.block_until_ready(E_var)
    energies_gpu.append(float(E_var))

end = time.perf_counter()

gpu_time = end - start

print("GPU runtime:", gpu_time, "seconds")
print("Final energy:", energies_gpu[-1])

In [ ]:
plt.figure(figsize=(7, 4))

plt.plot(energies_gpu, label="GPU VMC energy")

plt.xlabel("Iteration")
plt.ylabel("Variational energy")
plt.legend()
plt.grid(True)
plt.show()

Compare CPU vs GPU

In [ ]:
print("CPU runtime:", cpu_time)
print("GPU runtime:", gpu_time)
print("Speedup:", cpu_time / gpu_time)

Expected result: for small systems, GPU may not help much because compilation and transfer overhead are large. For larger systems, larger N_MC, and larger neural networks, GPU should become more useful.

In [ ]:
N_spins = 32
N_MC = 500
num_iterations = 200
hidden_dims = (64, 64)

In [ ]:
N_spins = 64
N_MC = 1000
num_iterations = 300
hidden_dims = (128, 128)